In [1]:
import pickle
import warnings
import json
import pandas as pd
from datetime import datetime, timedelta
from golf_helpers import load_event_odds, american_odds_to_implied_prob, estimate_finishing_position, calc_slope, recent_streak
warnings.filterwarnings('ignore')

folder_path = '/Users/holden.bridge/Desktop/golf-research/Golf Trends/owgr_historical'
with open(f'{folder_path}/owgr_dict.json', 'r') as f:
    owgr_dict = json.load(f)
with open(f'{folder_path}/tournament_fields.json', 'r') as f:
    tournament_fields = json.load(f)

xgb_model = pickle.load(open(f'beat_expectation_model_{datetime.now().strftime("%m_%d_26")}.pkl', 'rb'))

event_dates = {
    'WMOpen2026': '2026-02-07',
    'ATTPebble2026': '2026-02-14',
    'Genesis2026': '2026-02-21',
    'Cognizant2026': '2026-02-28',
}

In [ ]:

def evaluate_live_event(event_start_date, owgr_dict, event_name):
    weeks_before = range(1, 13)
    base_date = datetime.strptime(event_start_date, "%Y-%m-%d")
    week_dates = {}
    for w in weeks_before:
        target_date = base_date - timedelta(weeks=w)
        week_dates[f'Avg_Points_{w}weekbefore'] = target_date.strftime("%Y-%m-%d")

    player_odds = load_event_odds(event_name, folder_path)

    rows = []
    for player, data in owgr_dict.items():
        if player not in tournament_fields[event_name]:
            continue
        row = {'PlayerName': player}
        for colname, week_date in week_dates.items():
            row[colname] = data.get(week_date, {}).get('Avg_Points', None)
        row['Avg_Points_StartEvent'] = data.get(event_start_date, {}).get('Avg_Points', None)
        if row['Avg_Points_StartEvent'] is not None:
            row['EventName'] = event_name
            row['FinishingPosition'] = tournament_fields[event_name][player]['Finishing Position']
            odds = player_odds.get(player, {})
            row['WinOdds'] = odds.get('WinOdds', None)
            row['T5Odds'] = odds.get('T5Odds', None)
            row['T10Odds'] = odds.get('T10Odds', None)
            row['T20Odds'] = odds.get('T20Odds', None)
            rows.append(row)

    return pd.DataFrame(rows)

def prepare_live_event(df):
    df_result = df.copy()
    for i in range(1, 13):
        week_col = f'Avg_Points_{i}weekbefore'
        df_result[f'Week{i}Change'] = df_result['Avg_Points_StartEvent'] - df_result.get(week_col)
    # Also include Avg_Points_StartEvent and EventName in the resulting DataFrame
    df_result = df_result[[col for col in df_result.columns if ('Change' in col or col == 'PlayerName') or ('Odds' in col or col == 'PlayerName') or col == 'Avg_Points_StartEvent' or col == 'EventName' or col == 'FinishingPosition']]
    return df_result

def view_event_predictions(event):
    df_event = evaluate_live_event(event_dates[event], owgr_dict, event)
    df_modeling_event = prepare_live_event(df_event)
    df_model_event_clean = df_modeling_event.dropna()
    df_model_event_clean['T20Prob'] = df_model_event_clean['T20Odds'].apply(american_odds_to_implied_prob)
    df_model_event_clean['EstimatedFinishingPosition'] = df_model_event_clean.apply(estimate_finishing_position, axis=1)
    df_model_event_clean['momentum_slope'] = df_model_event_clean.apply(calc_slope, axis=1)
    df_model_event_clean['recent_momentum'] = df_model_event_clean[['Week1Change', 'Week2Change', 'Week3Change', 'Week4Change']].mean(axis=1)
    df_model_event_clean['distant_momentum'] = df_model_event_clean[['Week9Change', 'Week10Change', 'Week11Change', 'Week12Change']].mean(axis=1)
    df_model_event_clean['recent_vs_distant'] = df_model_event_clean['recent_momentum'] - df_model_event_clean['distant_momentum']
    df_model_event_clean['momentum_volatility'] = df_model_event_clean[[f'Week{i}Change' for i in range(1, 13)]].std(axis=1)
    df_model_event_clean['best_week'] = df_model_event_clean[[f'Week{i}Change' for i in range(1, 13)]].max(axis=1)
    df_model_event_clean['worst_week'] = df_model_event_clean[[f'Week{i}Change' for i in range(1, 13)]].min(axis=1)
    df_model_event_clean['week_range'] = df_model_event_clean['best_week'] - df_model_event_clean['worst_week']
    df_model_event_clean['positive_week_count'] = (df_model_event_clean[[f'Week{i}Change' for i in range(1, 13)]] > 0).sum(axis=1)
    df_model_event_clean['recent_positive_streak'] = df_model_event_clean.apply(recent_streak, axis=1)
    df_model_event_clean['acceleration_4v12'] = df_model_event_clean['Week4Change'] - df_model_event_clean['Week12Change']
    df_model_event_clean['points_x_momentum8'] = df_model_event_clean['Avg_Points_StartEvent'] * df_model_event_clean['Week8Change']
    df_model_event_clean['ModelPrediction'] = xgb_model.predict_proba(df_model_event_clean[['Avg_Points_StartEvent', 'Week4Change', 'recent_vs_distant', 'recent_positive_streak', 'acceleration_4v12', 'points_x_momentum8']])[:, 1]
    df_model_event_clean['T20_ImpliedProb'] = df_model_event_clean['T20Odds'].apply(american_odds_to_implied_prob)
    return df_model_event_clean[['PlayerName', 'ModelPrediction', 'T20Odds', 'T20_ImpliedProb', 'FinishingPosition', 'EstimatedFinishingPosition']].sort_values(by='ModelPrediction', ascending=False).head(5)


view_event_predictions('Genesis2026')

,PlayerName,ModelPrediction,T20Odds,T20_ImpliedProb,FinishingPosition,EstimatedFinishingPosition
31,Ryan Fox,0.674069,+410,0.196078,7,65
35,Akshay Bhatia,0.666023,+175,0.363636,16,40
45,Adam Scott,0.666023,+175,0.363636,4,40
14,Collin Morikawa,0.647566,+105,0.487805,7,20
42,Harry Hall,0.639120,+240,0.294118,999,40


In [3]:
view_event_predictions('Cognizant2026')

,PlayerName,ModelPrediction,T20Odds,T20_ImpliedProb,FinishingPosition,EstimatedFinishingPosition
1,Shane Lowry,0.666023,-115,0.534884,2,10
39,Austin Smotherman,0.627821,+500,0.166667,2,65
10,Michael Thorbjornsen,0.618915,+140,0.416667,999,20
13,Matt Wallace,0.614203,+290,0.256410,40,40
7,Daniel Berger,0.610382,+160,0.384615,32,40
